<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap07/7_2_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 7.2: Backpropagation**

This notebook runs the backpropagation algorithm on a deep neural network as described in section 7.4 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

---

**🇧🇷 Português**

# **Notebook 7.2: Retropropagação (Backpropagation)**

Este notebook executa o algoritmo de retropropagação (backpropagation) em uma rede neural profunda, conforme descrito na seção 7.4 do livro.

Trabalhe nas células abaixo, executando cada célula por vez. Em vários lugares você verá as palavras "TODO". Siga as instruções nesses pontos e faça previsões sobre o que vai acontecer ou escreva código para completar as funções.

Entre em contato em udlbookmail@gmail.com se encontrar algum erro ou tiver alguma sugestão.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

First let's define a neural network.  We'll just choose the weights and biases randomly for now

---

**🇧🇷 Português**

Primeiro, vamos definir uma rede neural. Por enquanto, vamos apenas escolher os pesos e os biases (viéses) de forma aleatória

In [2]:
# Set seed so we always get the same random numbers
# PT: Define a semente para sempre obtermos os mesmos números aleatórios
np.random.seed(0)

# Number of hidden layers
# PT: Número de camadas ocultas
K = 5
# Number of neurons per layer
# PT: Número de neurônios por camada
D = 6
# Input layer
# PT: Camada de entrada
D_i = 1
# Output layer
# PT: Camada de saída
D_o = 1

# Make empty lists
# PT: Cria listas vazias
all_weights = [None] * (K+1)
all_biases = [None] * (K+1)

# Create input and output layers
# PT: Cria as camadas de entrada e saída
all_weights[0] = np.random.normal(size=(D, D_i))
all_weights[-1] = np.random.normal(size=(D_o, D))
all_biases[0] = np.random.normal(size =(D,1))
all_biases[-1]= np.random.normal(size =(D_o,1))

# Create intermediate layers
# PT: Cria as camadas intermediárias
for layer in range(1,K):
  all_weights[layer] = np.random.normal(size=(D,D))
  all_biases[layer] = np.random.normal(size=(D,1))

In [3]:
# Define the Rectified Linear Unit (ReLU) function
# PT: Define a função Unidade Linear Retificada (ReLU)
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation

Now let's run our random network.  The weight matrices $\boldsymbol\Omega_{0\ldots K}$ are the entries of the list "all_weights" and the biases $\boldsymbol\beta_{0\ldots K}$ are the entries of the list "all_biases"

We know that we will need the preactivations $\mathbf{f}_{0\ldots K}$ and the activations $\mathbf{h}_{1\ldots K}$ for the forward pass of backpropagation, so we'll store and return these as well.


---

**🇧🇷 Português**

Agora vamos executar nossa rede aleatória. As matrizes de pesos $\boldsymbol\Omega_{0\ldots K}$ são os elementos da lista "all_weights" e os biases $\boldsymbol\beta_{0\ldots K}$ são os elementos da lista "all_biases"

Sabemos que vamos precisar das pré-ativações $\mathbf{f}_{0\ldots K}$ e das ativações $\mathbf{h}_{1\ldots K}$ para o passo para frente (forward pass) da retropropagação, então também vamos armazená-las e retorná-las.

In [4]:
def compute_network_output(net_input, all_weights, all_biases):

  # Retrieve number of layers
  # PT: Recupera o número de camadas
  K = len(all_weights) -1

  # We'll store the pre-activations at each layer in a list "all_f"
  # PT: Vamos armazenar as pré-ativações de cada camada em uma lista "all_f"
  # and the activations in a second list "all_h".
  # PT: e as ativações em uma segunda lista "all_h".
  all_f = [None] * (K+1)
  all_h = [None] * (K+1)

  #For convenience, we'll set
  # PT: Por conveniência, vamos definir
  # all_h[0] to be the input, and all_f[K] will be the output
  # PT: all_h[0] como a entrada, e all_f[K] será a saída
  all_h[0] = net_input

  # Run through the layers, calculating all_f[0...K-1] and all_h[1...K]
  # PT: Percorre as camadas, calculando all_f[0...K-1] e all_h[1...K]
  for layer in range(K):
      # Update preactivations and activations at this layer according to eqn 7.17
      # PT: Atualiza as pré-ativações e ativações nesta camada de acordo com a eq. 7.17
      # Remember to use np.matmul for matrix multiplications
      # PT: Lembre-se de usar np.matmul para multiplicações de matrizes
      all_f[layer] = all_biases[layer] + np.matmul(all_weights[layer],all_h[layer])
      all_h[layer+1] = ReLU(all_f[layer])

  # Compute the output from the last hidden layer
  # PT: Calcula a saída a partir da última camada oculta
  all_f[K] = all_biases[-1] + np.matmul(all_weights[-1], all_h[-1])

  # Retrieve the output
  # PT: Recupera a saída
  net_output = all_f[K]

  return net_output, all_f, all_h

In [5]:
# Define input
# PT: Define a entrada
net_input = np.ones((D_i,1)) * 1.2
# Compute network output
# PT: Calcula a saída da rede
net_output, all_f, all_h = compute_network_output(net_input,all_weights, all_biases)
print("True output = %3.3f, Your answer = %3.3f"%(1.907, net_output[0,0]))

True output = 1.907, Your answer = 1.907


Now let's define a loss function.  We'll just use the least squares loss function. We'll also write a function to compute dloss_doutput

---

**🇧🇷 Português**

Agora vamos definir uma função de perda (loss function). Vamos usar apenas a função de perda de mínimos quadrados (least squares). Também vamos escrever uma função para calcular dloss_doutput

In [6]:
def least_squares_loss(net_output, y):
  return np.sum((net_output-y) * (net_output-y))

def d_loss_d_output(net_output, y):
    return 2*(net_output -y);

In [7]:
y = np.ones((D_o,1)) * 20.0
loss = least_squares_loss(net_output, y)
print("y = %3.3f Loss = %3.3f"%(y[0,0], loss))

y = 20.000 Loss = 327.371


Now let's compute the derivatives of the network.  We already computed the forward pass.  Let's compute the backward pass.

---

**🇧🇷 Português**

Agora vamos calcular as derivadas da rede. Já calculamos o passo para frente (forward pass). Vamos calcular o passo para trás (backward pass).

In [8]:
# We'll need the indicator function
# PT: Vamos precisar da função indicadora
def indicator_function(x):
  x_in = np.array(x)
  x_in[x_in>0] = 1
  x_in[x_in<=0] = 0
  return x_in

# Main backward pass routine
# PT: Rotina principal do passo para trás (backward pass)
def backward_pass(all_weights, all_biases, all_f, all_h, y):
  # We'll store the derivatives dl_dweights and dl_dbiases in lists as well
  # PT: Vamos armazenar as derivadas dl_dweights e dl_dbiases em listas também
  all_dl_dweights = [None] * (K+1)
  all_dl_dbiases = [None] * (K+1)
  # And we'll store the derivatives of the loss with respect to the activation and preactivations in lists
  # PT: E vamos armazenar as derivadas da perda em relação à ativação e às pré-ativações em listas
  all_dl_df = [None] * (K+1)
  all_dl_dh = [None] * (K+1)
  # Again for convenience we'll stick with the convention that all_h[0] is the net input and all_f[k] in the net output
  # PT: Novamente por conveniência, vamos seguir a convenção de que all_h[0] é a entrada da rede e all_f[k] é a saída da rede

  # Compute derivatives of the loss with respect to the network output
  # PT: Calcula as derivadas da perda em relação à saída da rede
  all_dl_df[K] = np.array(d_loss_d_output(all_f[K],y))

  # Now work backwards through the network
  # PT: Agora percorre a rede de trás para frente
  for layer in range(K,-1,-1):
    # Calculate the derivatives of the loss with respect to the biases at layer from all_dl_df[layer]. (eq 7.22)
    # Calcula as derivadas da perda em relação aos biases na camada a partir de all_dl_df[layer]. (eq. 7.22)
    # To take a copy of matrix X, use Z=np.array(X)
    # PT: NOTA! Para fazer uma cópia da matriz X, use Z=np.array(X)
    all_dl_dbiases[layer] = np.array(all_dl_df[layer])

    # Calculate the derivatives of the loss with respect to the weights at layer from all_dl_df[layer] and all_h[layer] (eq 7.23)
    # Calcula as derivadas da perda em relação aos pesos na camada a partir de all_dl_df[layer] e all_h[layer] (eq. 7.23)
    # Don't forget to use np.matmul
    # PT: Não se esqueça de usar np.matmul
    all_dl_dweights[layer] = np.matmul(all_dl_df[layer], all_h[layer].transpose())

    # calculate the derivatives of the loss with respect to the activations from weight and derivatives of next preactivations (second part of last line of eq 7.25)
    # calcula as derivadas da perda em relação às ativações a partir dos pesos e das derivadas das próximas pré-ativações (segunda parte da última linha da eq. 7.25)
    all_dl_dh[layer] = np.matmul(all_weights[layer].transpose(), all_dl_df[layer])

    if layer > 0:
      # Calculate the derivatives of the loss with respect to the pre-activation f (use derivative of ReLu function, first part of last line of eq. 7.25)
      # Calcula as derivadas da perda em relação à pré-ativação f (use a derivada da função ReLU, primeira parte da última linha da eq. 7.25)
      all_dl_df[layer-1] = indicator_function(all_f[layer-1]) * all_dl_dh[layer]

  return all_dl_dweights, all_dl_dbiases

In [9]:
all_dl_dweights, all_dl_dbiases = backward_pass(all_weights, all_biases, all_f, all_h, y)

In [10]:
np.set_printoptions(precision=3)
# Make space for derivatives computed by finite differences
# PT: Reserva espaço para as derivadas calculadas por diferenças finitas
all_dl_dweights_fd = [None] * (K+1)
all_dl_dbiases_fd = [None] * (K+1)

# Let's test if we have the derivatives right using finite differences
# PT: Vamos testar se as derivadas estão corretas usando diferenças finitas
delta_fd = 0.000001

# Test the dervatives of the bias vectors
# PT: Testa as derivadas dos vetores de bias
for layer in range(K+1):
  dl_dbias  = np.zeros_like(all_dl_dbiases[layer])
  # For every element in the bias
  # PT: Para cada elemento do bias
  for row in range(all_biases[layer].shape[0]):
    # Take copy of biases  We'll change one element each time
    # PT: Faz uma cópia dos biases. Vamos alterar um elemento a cada vez
    all_biases_copy = [np.array(x) for x in all_biases]
    all_biases_copy[layer][row] += delta_fd
    network_output_1, *_ = compute_network_output(net_input, all_weights, all_biases_copy)
    network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
    dl_dbias[row] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dbiases_fd[layer] = np.array(dl_dbias)
  print("-----------------------------------------------")
  print("Bias %d, derivatives from backprop:"%(layer))
  print(all_dl_dbiases[layer])
  print("Bias %d, derivatives from finite differences"%(layer))
  print(all_dl_dbiases_fd[layer])
  if np.allclose(all_dl_dbiases_fd[layer],all_dl_dbiases[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")



# Test the derivatives of the weights matrices
# PT: Testa as derivadas das matrizes de pesos
for layer in range(K+1):
  dl_dweight  = np.zeros_like(all_dl_dweights[layer])
  # For every element in the bias
  # PT: Para cada elemento do bias
  for row in range(all_weights[layer].shape[0]):
    for col in range(all_weights[layer].shape[1]):
      # Take copy of biases  We'll change one element each time
      # PT: Faz uma cópia dos biases. Vamos alterar um elemento a cada vez
      all_weights_copy = [np.array(x) for x in all_weights]
      all_weights_copy[layer][row][col] += delta_fd
      network_output_1, *_ = compute_network_output(net_input, all_weights_copy, all_biases)
      network_output_2, *_ = compute_network_output(net_input, all_weights, all_biases)
      dl_dweight[row][col] = (least_squares_loss(network_output_1, y) - least_squares_loss(network_output_2,y))/delta_fd
  all_dl_dweights_fd[layer] = np.array(dl_dweight)
  print("-----------------------------------------------")
  print("Weight %d, derivatives from backprop:"%(layer))
  print(all_dl_dweights[layer])
  print("Weight %d, derivatives from finite differences"%(layer))
  print(all_dl_dweights_fd[layer])
  if np.allclose(all_dl_dweights_fd[layer],all_dl_dweights[layer],rtol=1e-05, atol=1e-08, equal_nan=False):
    print("Success!  Derivatives match.")
  else:
    print("Failure!  Derivatives different.")

-----------------------------------------------
Bias 0, derivatives from backprop:
[[ -4.486]
 [  4.947]
 [  6.812]
 [ -3.883]
 [-24.935]
 [  0.   ]]
Bias 0, derivatives from finite differences
[[ -4.486]
 [  4.947]
 [  6.812]
 [ -3.883]
 [-24.935]
 [  0.   ]]
Success!  Derivatives match.
-----------------------------------------------
Bias 1, derivatives from backprop:
[[ -0.   ]
 [-11.297]
 [  0.   ]
 [  0.   ]
 [-10.722]
 [  0.   ]]
Bias 1, derivatives from finite differences
[[  0.   ]
 [-11.297]
 [  0.   ]
 [  0.   ]
 [-10.722]
 [  0.   ]]
Success!  Derivatives match.
-----------------------------------------------
Bias 2, derivatives from backprop:
[[-0.   ]
 [-0.   ]
 [ 0.938]
 [ 0.   ]
 [-9.993]
 [ 0.508]]
Bias 2, derivatives from finite differences
[[ 0.   ]
 [ 0.   ]
 [ 0.938]
 [ 0.   ]
 [-9.993]
 [ 0.508]]
Success!  Derivatives match.
-----------------------------------------------
Bias 3, derivatives from backprop:
[[-0.   ]
 [-4.8  ]
 [-1.661]
 [-0.   ]
 [ 3.393]
 [ 5.391]